<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/transformers_agents_multiagents/KMeans_Text_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 This shows how thek sklearn api can be used to cluster documents by topics using a Bag of words approach. Latent Semantic Analysis is used to reduce dimensionality and discover patterns in the data

In [ ]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups

In [ ]:
categories = ["alt.atheism", "talk.religion.misc", "comp.graphics", "sci.space"]

In [ ]:
dataset = fetch_20newsgroups(
    remove=("headers", "footers", "quotes"), # remove metadata inorder to have a more sensible clustering problem
    subset="all",
    categories=categories,
    shuffle=True,
    random_state=42
)

In [ ]:
labels = dataset.target
unique_labels, category_sizes = np.unique(labels, return_counts=True)
true_k = unique_labels.shape[0]

print(f"{len(dataset.data)} documents - {true_k} categories")

3387 documents - 4 categories


### Quantifying the quality of clustering results
Let's define a fxn to score different clustering pipelines using several metrics.

Since we happen to have class labels for this specific dataset, it is possible to use evaluation metrics that leverage this "supervised" ground truth information to quantify the quality of the resulting clusters.
Examples of such metrics are:
1. Homogeneity: which quantifies how much clusters contain only members of a single class
2. Completeness: Which quantifies how much members of a given class are assigned to the same clusters;
3. V-measure: The harmonic mean of completeness and homogeneity.
4. Rand-Index: Which measures how frequently pairs of data points are grouped consistently according to the result of the clustering algorithm and the ground truth class assignment.
5. Adjusted Rand-Index: A chance-adjusted rand index such that random cluster assignment have an ARI of 0.0 in expectation.

If the ground truth labels are not known, evaluation can only be performed using the model results itself, in cases like this, the Silhouette Coefficient comes in handy.

In [ ]:
import numpy as np

np.mean(list({"s": -np.inf}.values()))

-inf

In [ ]:
def cluster_and_score(X, scores_dict, train_times_list):
    t0 = time()
    y_pred = kmeans.fit_predict(X)
    train_times_list.append(time() - t0)
    scores_dict["Homogeneity"].append(metrics.homogeneity_score(labels, y_pred))
    scores_dict["Completeness"].append(metrics.completeness_score(labels, y_pred))
    scores_dict["V-measure"].append(metrics.v_measure_score(labels, y_pred))
    scores_dict["Adjusted Rand-Index"].append(
        metrics.adjusted_rand_score(labels, y_pred)
    )
    scores_dict["Silhouette Coefficient"].append(
        metrics.silhouette_score(X, y_pred, sample_size=2000)
    )

In [ ]:
from collections import defaultdict
from time import time
from sklearn import metrics

evaluations = []
evaluations_std = []

def fit_and_evaluate(km, X, name=None, n_runs=5):
    name = km.__class__.__name__ if name is None else name

    train_times = []
    scores = defaultdict(list)
    for seed in range(n_runs):
        km.set_params(random_state=seed)
        # t0 = time()
        cluster_and_score(X, scores, train_times)
        # train_times.append(time() - t0)
        # scores["Homogeneity"].append(metrics.homogeneity_score(labels, km.labels_))
        # scores["Completeness"].append(metrics.completeness_score(labels, km.labels_))
        # scores["V-measure"].append(metrics.v_measure_score(labels, km.labels_))
        # scores["Adjusted Rand-Index"].append(
        #     metrics.adjusted_rand_score(labels, km.labels_)
        # )
        # scores["Silhouetter Coefficient"].append(
        #     metrics.silhouette_score(X, km.labels_, sample_size=2000)
        # )
    train_times = np.asarray(train_times)

    print(f"Clustering done in {train_times.mean():.2f} ± {train_times.std():.2f} s ")
    evaluation = {
        "estimator": name,
        "train_time": train_times.mean(),
    }
    evaluation_std = {
        "estimator": name,
        "train_time": train_times.std(),
    }
    for score_name, score_values in scores.items():
        mean_score, std_score = np.mean(score_values), np.std(score_values)
        print(f"{score_name}: {mean_score:.3f} ± {std_score:.3f}")
        evaluation[score_name] = mean_score
        evaluation_std[score_name] = std_score

    evaluations.append(evaluation)
    evaluations_std.append(evaluation_std)


Feature Extraction methods to be used in this example:
- TfidfVectorizer: Uses an in-memory vocabulary(a python dict) to map the most frequent words to feature indices and hence compute a word occurence frequency(sparse) matrix. The word frequencies are then reweighted using the inverse document frequency(idf) vector collected feature-wise over the corpus.
- HashingVector: Hashes word occurrences to a fixed dimensional space, possibly with collisions. The word count vectors are then normalized to each have l2-norm equal to one which seems important for k-means to work in high dimensional space.

From https://en.wikipedia.org/wiki/Bag-of-words_model:
A common alternative to using dictionaries(for a bag of words implementation) is to the hashing trick, where words are mapped directly to indicies with a hashing function thus no memory is required to store a dictionary. In practice, this simplifies the implementation of bag-of-words models and improves scalability. (TF-IDF Vectorizer is the dictionaries approach, Hashing vectorizer is the hashing trick approach)


It is possible to post-process these extracted features using dimensionality reduction.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_df=0.5, # ignore terms that appear in more than 50% of the documents.
    min_df=5, # ignore terms not present in atleast 5 documents
    stop_words="english",
)
t0 = time()
X_tfidf = vectorizer.fit_transform(dataset.data)

print(f"Vectorization done in {time() - t0:.3f} s")
print(f"n_samples: {X_tfidf.shape[0]}, n_features: {X_tfidf.shape[1]}")

Vectorization done in 1.412 s
n_samples: 3387, n_features: 7929


In [ ]:
# Quantify the sparsity of the X_tfidf matrix as the fraaction of non-zero entries
# divided by the total number of elements
print(f"{X_tfidf.nnz / np.prod(X_tfidf.shape):.3f}")

0.007


In [ ]:
# Around 0.7% of the entries of the X_tfidf matrix are non zero

### Clustering Sparse data with K-means

As both Kmeans and MiniBatchKMeans optimize a non-convex objective function, their clustering is not guaranteed to be optimal for a given random init. Even further, on sparse high dimensional data such as text vectorized using the Bag of Words approach, k-means can initialize centroids on extremely isolated data points. Those data points can stay their own centroids all along.

The below code illustrates how the previous phenomenon can sometimes lead to highly imbalaced clusters, depending on the random initialization

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
for seed in range(5):
    kmeans = KMeans(
        n_clusters=true_k,
        max_iter=100,
        n_init=1,
        random_state=seed
    ).fit(X_tfidf)
    cluster_ids, cluster_sizes = np.unique(kmeans.labels_, return_counts=True)
    print(f"Number of elements assigned to each cluster: {cluster_sizes}")

print("\nTrue number of documents in each category according to the class labels: "
      f"{category_sizes}")

Number of elements assigned to each cluster: [2050  711  180  446]
Number of elements assigned to each cluster: [1689  638  480  580]
Number of elements assigned to each cluster: [   1    1    1 3384]
Number of elements assigned to each cluster: [1887  311  332  857]
Number of elements assigned to each cluster: [1688  636  454  609]

True number of documents in each category according to the class labels: [799 973 987 628]


To avoid this problem, one possibility is to increase the number of runs with independent random initiations n_init. In such case, the clustering with the best inertia(objective function of k-means) is choosen.

In [ ]:
kmeans = KMeans(
    n_clusters=true_k, max_iter=100, n_init=5
)
fit_and_evaluate(kmeans, X_tfidf, name="KMeans\non tf-idf vectors")

Clustering done in 0.48 ± 0.08 s 
Homogeneity: 0.351 ± 0.006
Completeness: 0.402 ± 0.012
V-measure: 0.375 ± 0.008
Adjusted Rand-Index: 0.207 ± 0.015
Silhouette Coefficient: 0.007 ± 0.000


All the clustering evaluation metrics have a maximuvalue of 1.0(for a perfect clustering result). Higher values are better. values of the Adjusted Rand-Index close to 0.0 correspond to a random labelling. Notice from the scores above that the cluster assignment is indeed well above chance level, but the overall quality can certainly improve.

Keep in mind that the class labels may not reflect accurately the document topics and therefore metrics that use labels are not necessarily the best to evaluate the quality of our clustering pipeline.

### Performing Dimensionality reduction using LSA
A `n_init=1` can still be used as long as the dimension of the vectorized space is reduced first to make k-means more stable. For such purpose, we use `TruncatedSVD`, which works on term count/tf-idf matrices. Since SVD results are not normalized, we redo the normalization to improve the Kmeans result. Using SVD to reduce the dimensionality of TF-IDF document vectors is often known as Latent Semantic Analysis(LSA) in the information retrieval and text mining literature.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer

lsa = make_pipeline(TruncatedSVD(n_components=100), Normalizer(copy=False))
t0 = time()
X_lsa = lsa.fit_transform(X_tfidf)
explained_variance = lsa[0].explained_variance_ratio_.sum()

print(f"LSA done in {time() - t0:.3f} s")
print(f"Explained variance of the SVD step: {explained_variance * 100:.1f}%")

Using  a single initialization means the processing time will be reduced for both `KMeans` and `MiniBatchKMeans`

In [ ]:
kmeans = KMeans(
    n_clusters=true_k,
    max_iter=100,
    n_init=1
)
fit_and_evaluate(kmeans, X_lsa, name="KMeans\nwith LSA on tf-idf vectors")

We can observe that clustering on the LSA representation of the document is significantly faster(both because of n_init=1 and because the dimensionality of the LSA feature space is much smaller). Futhermore, all the clustering evaluation metrics have improved.  Lets repeat the experiment with `MiniBatchKMeans`

In [ ]:
from sklearn.cluster import MiniBatchKMeans

In [ ]:
minibatch_kmeans = MiniBatchKMeans(
    n_clusters=true_k,
    n_init=1,
    init_size=1000,
    batch_size=1000
)
fit_and_evaluate(minibatch_kmeans, X_lsa, name="MiniBatchKMeans\nwith LSA on tf-idf vectors")

### Top Terms per cluster
since `TfidfVectorizer` can be inverted, we can identify the cluster centers, which provide an intuition of the most influential words for each cluster.

In [ ]:
original_space_centroids = lsa[0].inverse_transform(kmeans.cluster_centers_)
order_centroids = original_space_centroids.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for i in range(true_k):
    print(f"Cluster {i}: ", end="")
    for ind in order_centroids[i, :10]:
        print(f"{terms[ind]} ", end="")
    print()

### HashingVectorizer
An alternative vectorization can be done using a `HashingVectorizer` instance, which does not provide IDF weighting as this is a stateless model(the fit method does nothing). When IDF weighting is needed, it can be added by pipelining the `HashingVectorizer` output to a `TfidfTransformer` instance. In this case, we also add LSA to the pipeline to reduce the dimension and sparsity of the hashed vector space.

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
lsa_vectorizer = make_pipeline(
    HashingVectorizer(stop_words="english", n_features=50_000),
    TfidfTransformer(),
    TruncatedSVD(n_components=100, random_state=0),
    Normalizer(copy=False),
)
t0 = time()
X_hashed_lsa = lsa_vectorizer.fit_transform(dataset.data)
print(f"vectorization done in {time() - t0:.3f} s")

One Can observe that the LSA step takes a relatively long time to fit, especially with hashed vectors. The reason is that a hashed space is typically large(set to `n_features=50_000` in this example). One can try lowering the number of features at the expnse of having a larger fraction of features with hash collisions as showing [here](https://scikit-learn.org/stable/auto_examples/text/plot_hashing_vs_dict_vectorizer.html#sphx-glr-auto-examples-text-plot-hashing-vs-dict-vectorizer-py)


We now fit and evaluate the `kmeans` and `minibatch_kmeans` instances on this hashed-lsa-reduced data

In [ ]:
fit_and_evaluate(kmeans, X_hashed_lsa, name="KMeans\nwith LSA on hashed vectors")

In [ ]:
fit_and_evaluate(minibatch_kmeans, X_hashed_lsa,
                 name="MiniBatchKMeans\nwith LSA on hashed vectors"
                )

Both methods lead to good results that are similar to running the same models on the traditional LSA vectors(without hashing).

### USing Sentence Transformer

### Clustering evaluation summary


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, (ax0, ax1) = plt.subplots(ncols=2, figsize=(16, 6), sharey=True)

df = pd.DataFrame(evaluations[::-1]).set_index("estimator")
df_std = pd.DataFrame(evaluations_std[::-1]).set_index("estimator")

df.drop(
    ["train_time"],
    axis="columns",
    ).plot.barh(ax=ax0, xerr=df_std)
ax0.set_xlabel("Clustering scores")
ax0.set_ylabel("")

df["train_time"].plot.barh(ax=ax1, xerr=df_std["train_time"])
ax1.set_xlabel("Clustering time (s)")
plt.tight_layout()

`KMeans` and `MiniBatchKMeans` suffer form the `Curse of Dimensionality` for high dimensional datasets such as text data. This is the reason why the overall scores improve when using LSA. Using LSA reduced data also improves the stability and requires lower clustering time, though keep in mind that the LSA step itself takes a long time, especially with hashed vectors.

The Silhouette coefficient is defined between 0 and 1. In all cases we obtain values close to 0(even if they improve a bit after using LSA) because its definition requires measuring distances, in contrast with other evaluation metrics such as the v-measure and the adjusted rand index which are only based on cluster assignments rather than distances. One should not compare the silhoette coefficient between spaces of different dimension, due to the different notions of distance they imply.

The homogeneity, completeness and v-measure metrics do not yield a baseline with regards to random labeling: this means that depending on the number of samples, clusters and ground truth clases, a completely random labeling will not alway yield same values.In particular, random labelling won't yield zero scores, especially when the number of clusters is large. This problem can be safely ignored when the number of samples is more than a thousand and the number of clusters is less than 10, which is the case of the present example. For smaller sample sizes or larger number of clusters, it is safer to use an adjusted index such as the Adjusted Rand Index(ARI). [see example](https://scikit-learn.org/stable/auto_examples/cluster/plot_adjusted_for_chance_measures.html#sphx-glr-auto-examples-cluster-plot-adjusted-for-chance-measures-py)

The size of the error bars show that `MiniBatchKMeans` is less stable than `KMeans` for this relatively small dataset. It is more interesting to use when the number of samples is much bigger, but it can come at the expense of a small degradation in clustering quality compared to the traditional k-means algorithm.